# Predict Customer Churn — XGBoost Tuned Multi-Seed + Feature Engineering

Full pipeline combining the best of all approaches:

1. **Feature Engineering** — Row statistics + domain interaction features on numeric columns
2. **Robust Optuna Tuning** — `xgb.cv()` inside each trial (not a single 80/20 split)
3. **Multi-Seed Ensemble** — 5 seeds × 10 folds = 50 models averaged for stable predictions
4. **GPU Support** — Auto-detects Kaggle GPU, local CUDA, or falls back to CPU
5. **Progress Bars** — `tqdm` on all long-running loops

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import optuna
from tqdm.auto import tqdm
import warnings
import subprocess
import os
import gc

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Environment Detection ────────────────────────────────────────────────────
IS_KAGGLE = os.path.exists('/kaggle/input')

def has_local_gpu():
    try:
        result = subprocess.run(['nvidia-smi'], capture_output=True, timeout=5)
        return result.returncode == 0
    except Exception:
        return False

USE_GPU = IS_KAGGLE or has_local_gpu()
DATA_DIR = '/kaggle/input/playground-series-s6e3/' if IS_KAGGLE else '../data/'

print(f"Environment : {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"GPU         : {'Enabled ✓' if USE_GPU else 'Disabled — using CPU'}")
print(f"Data dir    : {DATA_DIR}")
print(f"XGBoost ver : {xgb.__version__}")

# ── Experiment Settings ───────────────────────────────────────────────────────
RUN_TUNING  = True   # False → skip Optuna, use PRECOMPUTED_PARAMS
N_TRIALS    = 50     # Increase to 100 on Kaggle GPU
N_CV_FOLDS  = 5      # Folds inside each Optuna trial
SEEDS       = [42, 2024, 777, 1337, 999]
N_SPLITS    = 10     # Folds per seed in ensemble
TOTAL_MODELS = len(SEEDS) * N_SPLITS

# Pre-computed fallback (used when RUN_TUNING = False)
PRECOMPUTED_PARAMS = {
    'learning_rate':     0.015,
    'max_depth':         8,
    'min_child_weight':  5,
    'subsample':         0.80,
    'colsample_bytree':  0.70,
    'colsample_bylevel': 0.80,
    'gamma':             0.10,
    'reg_alpha':         0.05,
    'reg_lambda':        0.50,
}

## 1. Feature Engineering

In [ ]:
def engineer_features(df):
    """Row-level statistics + domain interaction features on numeric columns."""
    df = df.copy()
    num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

    # Row aggregations
    df['num_sum']  = df[num_cols].sum(axis=1)
    df['num_mean'] = df[num_cols].mean(axis=1)
    df['num_std']  = df[num_cols].std(axis=1)
    df['num_max']  = df[num_cols].max(axis=1)
    df['num_min']  = df[num_cols].min(axis=1)

    # Domain interactions — captures billing drift over tenure
    df['Average_Monthly_Actual'] = df['TotalCharges'] / (df['tenure'] + 1e-5)
    df['Monthly_diff']  = df['MonthlyCharges'] - df['Average_Monthly_Actual']
    df['Monthly_ratio'] = df['MonthlyCharges'] / (df['Average_Monthly_Actual'] + 1e-5)

    return df

print("Feature engineering function defined.")

## 2. Load & Preprocess Data

In [ ]:
print(f"Loading data from {DATA_DIR} ...")
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
sub      = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))

TARGET = 'Churn'
if train_df[TARGET].dtype == 'object':
    train_df[TARGET] = train_df[TARGET].map({'Yes': 1, 'No': 0, '1': 1, '0': 0})

# ── Unified preprocessing on train + test ────────────────────────────────────
train_df['is_train'] = 1
test_df['is_train']  = 0
df = pd.concat([train_df, test_df], ignore_index=True)

features = [c for c in train_df.columns if c not in ['id', TARGET, 'is_train']]
categorical_features = []

print("Label encoding categorical features...")
for col in tqdm(features, desc='Encoding'):
    if df[col].dtype == 'object':
        categorical_features.append(col)
        le = LabelEncoder()
        df[col] = df[col].fillna('Missing')
        df[col] = le.fit_transform(df[col].astype(str))

num_features = [c for c in features if c not in categorical_features]
for col in num_features:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

train_enc = df[df['is_train'] == 1].reset_index(drop=True)
test_enc  = df[df['is_train'] == 0].reset_index(drop=True)

# ── Feature Engineering ───────────────────────────────────────────────────────
print("\nEngineering features...")
X      = engineer_features(train_enc[features])
X_test = engineer_features(test_enc[features])
y      = train_enc[TARGET]

# Cast categoricals for XGBoost native support
for col in categorical_features:
    X[col]      = X[col].astype('category')
    X_test[col] = X_test[col].astype('category')

print(f"\nTrain shape : {X.shape}")
print(f"Test shape  : {X_test.shape}")
print(f"Target dist : {y.value_counts().to_dict()}")
print(f"New features: {[c for c in X.columns if c not in features]}")

## 3. Optuna Hyperparameter Tuning

`xgb.cv()` runs a full 5-fold CV **per trial** so parameters are robust — not overfitted to a single validation split.

In [ ]:
# Base params shared across tuning and ensemble
base_params = {
    'objective':   'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'verbosity':   0,
    'seed':        42,
}
if USE_GPU:
    base_params['device'] = 'cuda'
    print("GPU params injected: device=cuda")

dtrain_full = xgb.DMatrix(X, label=y, enable_categorical=True)

if RUN_TUNING:
    print(f"\nStarting Optuna tuning: {N_TRIALS} trials × {N_CV_FOLDS}-fold CV each")

    pbar = tqdm(total=N_TRIALS, desc='Optuna Trials', unit='trial')
    best_value_so_far = [0.0]

    def objective(trial):
        params = base_params.copy()
        params.update({
            'learning_rate':     trial.suggest_float('learning_rate',     0.005, 0.1,  log=True),
            'max_depth':         trial.suggest_int(  'max_depth',         3,     10),
            'min_child_weight':  trial.suggest_int(  'min_child_weight',  1,     20),
            'subsample':         trial.suggest_float('subsample',         0.5,   1.0),
            'colsample_bytree':  trial.suggest_float('colsample_bytree',  0.3,   1.0),
            'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.3,   1.0),
            'gamma':             trial.suggest_float('gamma',             1e-8,  5.0,  log=True),
            'reg_alpha':         trial.suggest_float('reg_alpha',         1e-8,  10.0, log=True),
            'reg_lambda':        trial.suggest_float('reg_lambda',        1e-8,  10.0, log=True),
        })

        cv_result = xgb.cv(
            params,
            dtrain_full,
            num_boost_round=1500,
            nfold=N_CV_FOLDS,
            stratified=True,
            metrics='auc',
            early_stopping_rounds=50,
            verbose_eval=False,
            seed=42,
        )

        score = cv_result['test-auc-mean'].max()
        best_round = int(cv_result['test-auc-mean'].idxmax())
        trial.set_user_attr('best_round', best_round)

        # Update progress bar
        if score > best_value_so_far[0]:
            best_value_so_far[0] = score
        pbar.set_postfix({'AUC': f'{score:.5f}', 'Best': f'{best_value_so_far[0]:.5f}'})
        pbar.update(1)
        return score

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=N_TRIALS)
    pbar.close()

    best_tuned = study.best_trial
    best_round = best_tuned.user_attrs.get('best_round', 1000)

    print(f"\n{'='*55}")
    print(f"Best CV AUC  : {best_tuned.value:.5f}")
    print(f"Best round   : {best_round}")
    print(f"Best params  :")
    for k, v in best_tuned.params.items():
        print(f"  {k:25s}: {v}")
    print(f"{'='*55}")

    best_params = {**base_params, **best_tuned.params}

    # Save all trial results
    study.trials_dataframe().to_csv('xgboost_tuning_results.csv', index=False)
    print("\nAll trial results saved → xgboost_tuning_results.csv")

else:
    print("Skipping tuning — using PRECOMPUTED_PARAMS.")
    best_params = {**base_params, **PRECOMPUTED_PARAMS}
    best_round  = 1000
    for k, v in PRECOMPUTED_PARAMS.items():
        print(f"  {k:25s}: {v}")

## 4. Multi-Seed Ensemble

Trains `len(SEEDS) × N_SPLITS` models and averages predictions to reduce variance.

In [ ]:
print(f"Multi-Seed Ensemble: {len(SEEDS)} seeds × {N_SPLITS} folds = {TOTAL_MODELS} models\n")

dtest = xgb.DMatrix(X_test, enable_categorical=True)

test_preds_sum = np.zeros(len(X_test))
global_oof_sum = np.zeros(len(X))
fold_auc_log   = []

outer_bar = tqdm(SEEDS, desc='Seeds', position=0)

for seed in outer_bar:
    outer_bar.set_description(f'Seed {seed}')
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    seed_oof = np.zeros(len(X))

    inner_bar = tqdm(
        enumerate(skf.split(X, y)),
        total=N_SPLITS,
        desc=f'  Folds',
        position=1,
        leave=False,
    )

    for fold, (train_idx, val_idx) in inner_bar:
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        dtrain_fold = xgb.DMatrix(X_tr,  label=y_tr,  enable_categorical=True)
        dval_fold   = xgb.DMatrix(X_val, label=y_val, enable_categorical=True)

        fold_params = {**best_params, 'seed': seed}

        model = xgb.train(
            fold_params,
            dtrain_fold,
            num_boost_round=best_round + 200,  # headroom above best CV round
            evals=[(dval_fold, 'val')],
            early_stopping_rounds=150,
            verbose_eval=False,
        )

        itr = (0, model.best_iteration + 1)
        val_preds  = model.predict(dval_fold, iteration_range=itr)
        test_preds = model.predict(dtest,     iteration_range=itr)

        fold_auc = roc_auc_score(y_val, val_preds)
        fold_auc_log.append(fold_auc)

        seed_oof[val_idx]    = val_preds
        global_oof_sum[val_idx] += val_preds
        test_preds_sum       += test_preds

        inner_bar.set_postfix({'fold_auc': f'{fold_auc:.5f}'})

        del model, dtrain_fold, dval_fold
        gc.collect()

    seed_auc = roc_auc_score(y, seed_oof)
    outer_bar.set_postfix({'seed_oof_auc': f'{seed_auc:.5f}'})

# ── Final metrics ─────────────────────────────────────────────────────────────
global_oof  = global_oof_sum / len(SEEDS)
final_auc   = roc_auc_score(y, global_oof)

print(f"\n{'='*55}")
print(f"Fold AUC  — mean: {np.mean(fold_auc_log):.5f}  std: {np.std(fold_auc_log):.5f}")
print(f"Global Ensemble OOF AUC : {final_auc:.5f}")
print(f"{'='*55}")

## 5. Generate Submission

In [ ]:
final_test_preds = test_preds_sum / TOTAL_MODELS
sub[TARGET] = final_test_preds

out_file = 'submission_xgboost_tuned_multiseed_fe.csv'
sub.to_csv(out_file, index=False)

print(f"Submission saved → {out_file}")
print(f"Prediction range: [{final_test_preds.min():.4f}, {final_test_preds.max():.4f}]")
display(sub.head())